<a href="https://colab.research.google.com/github/na2003-gif/Bigdata_Project_2025.1/blob/main/Sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA

## Load data firm

In [2]:
from google.colab import files
uploaded = files.upload()

Saving FFIEC_Selected.zip to FFIEC_Selected (1).zip


In [3]:
!unzip "/content/FFIEC_Selected.zip"

Archive:  /content/FFIEC_Selected.zip
   creating: FFIEC_Selected/
  inflating: __MACOSX/._FFIEC_Selected  
   creating: FFIEC_Selected/2025Q4/
   creating: FFIEC_Selected/2025Q3/
   creating: FFIEC_Selected/2023Q1/
   creating: FFIEC_Selected/2021Q2/
   creating: FFIEC_Selected/2025Q2/
   creating: FFIEC_Selected/2021Q4/
   creating: FFIEC_Selected/2021Q3/
   creating: FFIEC_Selected/2014Q3/
   creating: FFIEC_Selected/2014Q4/
   creating: FFIEC_Selected/2018Q1/
   creating: FFIEC_Selected/2016Q1/
   creating: FFIEC_Selected/2020Q1/
   creating: FFIEC_Selected/2022Q2/
   creating: FFIEC_Selected/2022Q4/
   creating: FFIEC_Selected/2022Q3/
   creating: FFIEC_Selected/2024Q1/
   creating: FFIEC_Selected/2019Q2/
   creating: FFIEC_Selected/2017Q3/
   creating: FFIEC_Selected/2017Q4/
   creating: FFIEC_Selected/2019Q3/
   creating: FFIEC_Selected/2019Q4/
   creating: FFIEC_Selected/2015Q1/
   creating: FFIEC_Selected/2017Q2/
   creating: FFIEC_Selected/2023Q2/
   creating: FFIEC_Selected/

In [4]:
!ls "/content/FFIEC_Selected.zip"

/content/FFIEC_Selected.zip


## Data fed

In [6]:
import pandas as pd
df = pd.read_csv('/content/FEDFUNDS.csv')

# Processing

## Check cột

In [34]:
import os
import pandas as pd

def check_quarter_codes(data_path, codes):
    results = {}
    codes_set = set(codes)

    for quarter in os.listdir(data_path):
        q_path = os.path.join(data_path, quarter)
        if not os.path.isdir(q_path):
            continue

        code_file_map = {}

        for file in os.listdir(q_path):
            if not file.endswith(".txt"):
                continue
            if len(code_file_map) == len(codes_set):
                break

            filepath = os.path.join(q_path, file)

            try:
                # Only read headers to check columns
                df = pd.read_csv(filepath, sep=None, engine="python", nrows=0)
                cols = set(df.columns)
                remaining_codes = codes_set - set(code_file_map.keys())

                for c in remaining_codes:
                    if c in cols:
                        code_file_map[c] = filepath
            except:
                continue

        missing = list(codes_set - set(code_file_map.keys()))
        if missing:
            print(f"Quarter {quarter} is missing these codes: {', '.join(missing)}")

        results[quarter] = {
            "missing": missing,
            "complete": len(missing) == 0,
            "code_map": code_file_map  # Added this line to fix the KeyError
        }

    return results

In [28]:
check_quarter_codes("/content/FFIEC_Selected",
                    codes=["RIAD0093",
"RIAD4340",
"RIAD4074",
"RCON2170",
"RCON2200",
"RCON3210",
"RIAD4073",
"RIAD4107",
"RCON2948"])

{'2018Q1': {'missing': [], 'complete': True},
 '2014Q3': {'missing': [], 'complete': True},
 '2019Q4': {'missing': [], 'complete': True},
 '2022Q1': {'missing': [], 'complete': True},
 '2025Q1': {'missing': [], 'complete': True},
 '2016Q2': {'missing': [], 'complete': True},
 '2015Q4': {'missing': [], 'complete': True},
 '2022Q2': {'missing': [], 'complete': True},
 '2015Q2': {'missing': [], 'complete': True},
 '2020Q2': {'missing': [], 'complete': True},
 '2020Q4': {'missing': [], 'complete': True},
 '2022Q4': {'missing': [], 'complete': True},
 '2024Q2': {'missing': [], 'complete': True},
 '2020Q3': {'missing': [], 'complete': True},
 '2025Q2': {'missing': [], 'complete': True},
 '2017Q4': {'missing': [], 'complete': True},
 '2019Q2': {'missing': [], 'complete': True},
 '2018Q3': {'missing': [], 'complete': True},
 '2021Q2': {'missing': [], 'complete': True},
 '2023Q3': {'missing': [], 'complete': True},
 '2025Q3': {'missing': [], 'complete': True},
 '2025Q4': {'missing': [], 'comple

## Merge firm

In [35]:
import pandas as pd
import os

def extract_ffiec_panel(check_result, id_col="IDRSSD"):
    all_quarter_data = []

    for quarter, info in check_result.items():
        if not info["complete"]:
            continue

        merged_df = None


        code_map = info["code_map"]

        file_code_map = {}
        for c, filepath in code_map.items():
            file_code_map.setdefault(filepath, []).append(c)

        for filepath, code_list in file_code_map.items():
            try:
                df = pd.read_csv(
                    filepath,
                    sep=None,
                    engine="python",
                    encoding="latin1"
                )

                if id_col not in df.columns:
                    continue

                cols_keep = [id_col] + code_list
                df = df[cols_keep]

                df = df.drop_duplicates(subset=[id_col])

                if merged_df is None:
                    merged_df = df
                else:
                    merged_df = pd.merge(
                        merged_df,
                        df,
                        on=id_col,
                        how="outer"
                    )

            except Exception as e:
                print("Skip:", filepath)

        if merged_df is not None:
            merged_df["quarter"] = quarter
            all_quarter_data.append(merged_df)

    final_df = pd.concat(all_quarter_data, ignore_index=True)

    return final_df

In [36]:

check_results = check_quarter_codes("/content/FFIEC_Selected",
                    codes=["RIAD0093", "RIAD4340", "RIAD4074", "RCON2170", "RCON2200", "RCON3210", "RIAD4073", "RIAD4107", "RCON2948"])
final_df = extract_ffiec_panel(check_results)

In [39]:

final_df.to_csv('final_df1.csv', index=False)

# Merge Fed

In [41]:
def add_fed_funds(final_df, effr_path, method="mean"):
    _, file_extension = os.path.splitext(effr_path)
    if file_extension.lower() == '.xlsx':
        df = pd.read_excel(effr_path)
    elif file_extension.lower() == '.csv':
        df = pd.read_csv(effr_path)
    else:
        raise ValueError("Unsupported file format. Please provide an .xlsx or .csv file.")
    print("Columns in FEDFUNDS file:", df.columns.tolist())
    print("First 5 rows of FEDFUNDS file:\n", df.head())

    df.rename(columns={'FEDFUNDS': 'EFFR'}, inplace=True)
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df["quarter"] = df["observation_date"].dt.to_period("Q").astype(str)

    # Gộp theo quý và tính trung bình EFFR cho mỗi quý
    if method == "mean":
        effr_q = df.groupby("quarter")["EFFR"].mean().reset_index()
    elif method == "last":
        effr_q = df.groupby("quarter")["EFFR"].last().reset_index()
    elif method == "median":
        effr_q = df.groupby("quarter")["EFFR"].median().reset_index()
    else:
        raise ValueError("method phải là: mean / last / median")
    final_df["quarter"] = final_df["quarter"].astype(str).str.upper()
    effr_q["quarter"] = effr_q["quarter"].str.upper()
    if "EFFR" in final_df.columns:
        final_df = final_df.drop(columns=["EFFR"])

    # Merge final_df với effr_q (EFFR theo quý)
    final_df = pd.merge(
        final_df,
        effr_q,
        on="quarter",
        how="left"
    )

    return final_df

In [46]:
final_df = add_fed_funds(final_df, '/content/FEDFUNDS.csv', method="mean")
 #method: 'mean', 'last', 'median'
print(final_df.head())

Columns in FEDFUNDS file: ['observation_date', 'FEDFUNDS']
First 5 rows of FEDFUNDS file:
   observation_date  FEDFUNDS
0       2010-01-01      0.11
1       2010-02-01      0.13
2       2010-03-01      0.16
3       2010-04-01      0.20
4       2010-05-01      0.20
   IDRSSD RIAD4107 RIAD4340 RIAD4074 RIAD4073 RIAD0093 RCON2948 RCON2170  \
0    37.0      830      176      755       75        8    67195    81677   
1   242.0      425      168      369       56       13    38434    43646   
2   279.0     1980      279     1548      432       58   187680   216812   
3   354.0       80      -11       78        2        0     7281     8616   
4   457.0      594      205      534       60       22    43452    48513   

  RCON3210 RCON2200 quarter      EFFR  
0    14482    67083  2018Q1  1.446667  
1     5212    33916  2018Q1  1.446667  
2    29132   169480  2018Q1  1.446667  
3     1335     7279  2018Q1  1.446667  
4     5061    37392  2018Q1  1.446667  


In [47]:
print(final_df.isna().sum())

IDRSSD        46
RIAD4107       0
RIAD4340       0
RIAD4074       0
RIAD4073       0
RIAD0093       0
RCON2948    3666
RCON2170    3666
RCON3210    3666
RCON2200       0
quarter        0
EFFR           0
dtype: int64


In [50]:
final_df.to_csv('hi',index=True)

ValueError: No engine for filetype: ''

# Int Exp Beta

##In Exp/Asset

In [51]:
import pandas as pd
import statsmodels.api as sm

def run_bank_regressions(data, bank_id, y, x, add_const=True, robust=None, min_obs=8, print_progress=False):
    """
    Chạy OLS riêng cho từng bank.

    Parameters
    ----------
    data : pd.DataFrame
    bank_id : str
        IDRSSD
    y : str
        Biến phụ thuộc
    x : str | list[str]
        Biến độc lập
    add_const : bool
        Có thêm intercept hay không
    robust : None | str
       'HC3'
    min_obs : int
        Số quan sát tối thiểu để chạy 1 regression
    print_progress : bool
        Có in tiến trình hay không

    Returns
    -------
    results_df : pd.DataFrame
        Bảng kết quả từng bank
    """
    if isinstance(x, str):
        x = [x]

    cols = [bank_id, y] + x
    df = data[cols].copy()

    # ép numeric cho y và x
    for col in [y] + x:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=[y] + x)

    results = []

    for bank, g in df.groupby(bank_id):
        g = g.copy()

        if len(g) < min_obs:
            continue

        X = g[x]
        Y = g[y]

        if add_const:
            X = sm.add_constant(X)

        try:
            model = sm.OLS(Y, X).fit(cov_type=robust) if robust else sm.OLS(Y, X).fit()

            row = {
                bank_id: bank,
                'n_obs': int(model.nobs),
                'r2': model.rsquared,
                'adj_r2': model.rsquared_adj,
                'f_stat': model.fvalue,
                'f_pvalue': model.f_pvalue
            }

            # lưu coef, se, t, p cho từng biến
            for var in model.params.index:
                row[f'coef_{var}'] = model.params[var]
                row[f'se_{var}'] = model.bse[var]
                row[f't_{var}'] = model.tvalues[var]
                row[f'p_{var}'] = model.pvalues[var]

            results.append(row)

            if print_progress:
                print(f"Done bank {bank}")

        except Exception as e:
            if print_progress:
                print(f"Skip bank {bank}: {e}")

    results_df = pd.DataFrame(results)
    return results_df

def run_regression(data, y, x, add_const=True, robust=None, print_summary=True):
    if isinstance(x, str):
        x = [x]

    cols = [y] + x
    df = data[cols].copy()

    # ép toàn bộ về numeric, lỗi thì chuyển thành NaN
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna()

    X = df[x]
    Y = df[y]

    if add_const:
        X = sm.add_constant(X)

    model = sm.OLS(Y, X).fit(cov_type=robust) if robust else sm.OLS(Y, X).fit()

    if print_summary:
        print(model.summary())

    return model

In [56]:
import pandas as pd
import numpy as np

# Using final_df which contains the bank call report data
final_df["totalinexp"] = pd.to_numeric(final_df["RIAD4073"], errors="coerce")
final_df["avg_assets"] = pd.to_numeric(final_df["RCON2170"], errors="coerce")

final_df["avg_assets"] = final_df["avg_assets"].replace(0, np.nan)

# annualized interest expense rate
final_df["IntExpRate"] = 4 * (final_df["totalinexp"] / final_df["avg_assets"])

print(final_df[["RIAD4073", "avg_assets", "IntExpRate"]].head())

  RIAD4073  avg_assets  IntExpRate
0       75     81677.0    0.003673
1       56     43646.0    0.005132
2      432    216812.0    0.007970
3        2      8616.0    0.000929
4       60     48513.0    0.004947


In [58]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# =========================
# 1. Chuẩn bị dữ liệu
# =========================
# Fix: Use final_df which contains 'IntExpRate' and bank IDs
df_reg = final_df.copy()

# ép numeric
df_reg["IntExpRate"] = pd.to_numeric(df_reg["IntExpRate"], errors="coerce")
df_reg["EFFR"] = pd.to_numeric(df_reg["EFFR"], errors="coerce")

# sắp xếp
df_reg = df_reg.sort_values(["IDRSSD", "quarter"]).reset_index(drop=True)

# =========================
# 2. Tạo ΔIntExpRate theo từng bank
# =========================
df_reg["d_IntExpRate"] = df_reg.groupby("IDRSSD")["IntExpRate"].diff()

# =========================
# 3. Tạo ΔEFFR theo quý
# =========================
effr_q = (
    df_reg[["quarter", "EFFR"]]
    .drop_duplicates(subset=["quarter"])
    .sort_values("quarter")
    .reset_index(drop=True)
)

effr_q["d_EFFR"] = effr_q["EFFR"].diff()
effr_q["d_EFFR_l1"] = effr_q["d_EFFR"].shift(1)
effr_q["d_EFFR_l2"] = effr_q["d_EFFR"].shift(2)
effr_q["d_EFFR_l3"] = effr_q["d_EFFR"].shift(3)

# merge lại vào df_reg
df_reg = df_reg.drop(columns=["d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"], errors="ignore")
df_reg = df_reg.merge(
    effr_q[["quarter", "d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"]],
    on="quarter",
    how="left"
)

# =========================
# 4. Chạy OLS riêng cho từng bank
# =========================
results = []

for bank, g in df_reg.groupby("IDRSSD"):
    temp = g[[
        "d_IntExpRate",
        "d_EFFR",
        "d_EFFR_l1",
        "d_EFFR_l2",
        "d_EFFR_l3"
    ]].dropna().copy()

    if len(temp) < 8:
        continue

    if temp["d_IntExpRate"].nunique() < 2:
        continue

    X = temp[["d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"]]
    y = temp["d_IntExpRate"]
    X = sm.add_constant(X)

    try:
        model = sm.OLS(y, X).fit()
        b0 = model.params.get("d_EFFR", 0)
        b1 = model.params.get("d_EFFR_l1", 0)
        b2 = model.params.get("d_EFFR_l2", 0)
        b3 = model.params.get("d_EFFR_l3", 0)
        expense_beta = b0 + b1 + b2 + b3

        results.append({
            "IDRSSD": bank,
            "n_obs": len(temp),
            "expense_beta": expense_beta,
            "r2": model.rsquared
        })
    except:
        continue

expense_beta_df = pd.DataFrame(results)
print(expense_beta_df.head())
print("Số bank estimate được:", len(expense_beta_df))

   IDRSSD  n_obs  expense_beta        r2
0    37.0     42      0.001466  0.097030
1   242.0     42      0.004621  0.098409
2   279.0     42      0.005120  0.110062
3   354.0     42      0.008336  0.070033
4   457.0     42      0.004217  0.107830
Số bank estimate được: 5783


In [79]:
print(expense_beta_df["expense_beta"].describe().round(6))

count    5783.000000
mean        0.004033
std         0.012388
min        -0.151759
25%         0.002076
50%         0.004257
75%         0.006596
max         0.405310
Name: expense_beta, dtype: float64


In [65]:
if (expense_beta_df["expense_beta"] < 0).any():
    print((expense_beta_df["expense_beta"] < 0).sum())

672


##In Ex / Liability

In [68]:
import pandas as pd
import numpy as np

# Using final_df which contains the bank call report data
final_df["totalinexp"] = pd.to_numeric(final_df["RIAD4073"], errors="coerce")
final_df["avg_liab"] = pd.to_numeric(final_df["RCON2948"], errors="coerce")

final_df["avg_liab"] = final_df["avg_liab"].replace(0, np.nan)

# annualized interest expense rate
final_df["IntExpRatebylia"] = 4 * (final_df["totalinexp"] / final_df["avg_liab"])

print(final_df[["RIAD4073", "avg_liab", "IntExpRatebylia"]].head())

  RIAD4073  avg_liab  IntExpRatebylia
0       75   67195.0         0.004465
1       56   38434.0         0.005828
2      432  187680.0         0.009207
3        2    7281.0         0.001099
4       60   43452.0         0.005523


In [76]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# =========================
# 1. Chuẩn bị dữ liệu
# =========================
df_reg = final_df.copy()

# chọn đúng biến phụ thuộc
y_col = "IntExpRatebylia"

# ép kiểu
df_reg["IDRSSD"] = pd.to_numeric(df_reg["IDRSSD"], errors="coerce")
df_reg[y_col] = pd.to_numeric(df_reg[y_col], errors="coerce")
df_reg["EFFR"] = pd.to_numeric(df_reg["EFFR"], errors="coerce")

# bỏ dòng thiếu key
df_reg = df_reg.dropna(subset=["IDRSSD", "quarter", y_col, "EFFR"]).copy()

# nếu quarter là dạng 2014Q3 thì tách để sort đúng
df_reg["year"] = df_reg["quarter"].astype(str).str[:4].astype(int)
df_reg["qnum"] = df_reg["quarter"].astype(str).str[-1].astype(int)

df_reg = df_reg.sort_values(["IDRSSD", "year", "qnum"]).reset_index(drop=True)

# =========================
# 2. Tạo biến sai phân theo từng bank
# =========================
df_reg["d_IntExpRatebylia"] = df_reg.groupby("IDRSSD")[y_col].diff()

# =========================
# 3. Tạo ΔEFFR theo quý + 3 độ trễ
# =========================
effr_q = (
    df_reg[["quarter", "year", "qnum", "EFFR"]]
    .drop_duplicates(subset=["quarter"])
    .sort_values(["year", "qnum"])
    .reset_index(drop=True)
)

effr_q["d_EFFR"] = effr_q["EFFR"].diff()
effr_q["d_EFFR_l1"] = effr_q["d_EFFR"].shift(1)
effr_q["d_EFFR_l2"] = effr_q["d_EFFR"].shift(2)
effr_q["d_EFFR_l3"] = effr_q["d_EFFR"].shift(3)

df_reg = df_reg.drop(columns=["d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"], errors="ignore")

df_reg = df_reg.merge(
    effr_q[["quarter", "d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"]],
    on="quarter",
    how="left"
)

# =========================
# 4. Chạy OLS riêng cho từng bank
# =========================
results = []

for bank, g in df_reg.groupby("IDRSSD"):
    temp = g[
        ["d_IntExpRatebylia", "d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"]
    ].dropna().copy()

    # tối thiểu phải đủ quan sát
    if len(temp) < 8:
        continue

    # biến phụ thuộc phải có biến thiên
    if temp["d_IntExpRatebylia"].nunique() < 2:
        continue

    # EFFR changes cũng phải có biến thiên
    if temp[["d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"]].nunique().sum() < 4:
        continue

    X = temp[["d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"]]
    y = temp["d_IntExpRatebylia"]
    X = sm.add_constant(X)

    try:
        model = sm.OLS(y, X).fit()

        b0 = model.params.get("d_EFFR", np.nan)
        b1 = model.params.get("d_EFFR_l1", np.nan)
        b2 = model.params.get("d_EFFR_l2", np.nan)
        b3 = model.params.get("d_EFFR_l3", np.nan)

        expense_beta_2 = b0 + b1 + b2 + b3

        results.append({
            "IDRSSD": bank,
            "n_obs": len(temp),
            "coef_d_EFFR": b0,
            "coef_d_EFFR_l1": b1,
            "coef_d_EFFR_l2": b2,
            "coef_d_EFFR_l3": b3,
            "expense_beta_2": expense_beta_2,
            "r2": model.rsquared,
            "f_stat": model.fvalue,
            "f_pvalue": model.f_pvalue
        })

    except Exception:
        continue

expense_beta_df_2 = pd.DataFrame(results)

print(expense_beta_df_2.head())
print("Số bank estimate được:", len(expense_beta_df_2))

   IDRSSD  n_obs  coef_d_EFFR  coef_d_EFFR_l1  coef_d_EFFR_l2  coef_d_EFFR_l3  \
0    37.0     42    -0.004157        0.010268       -0.010827        0.006469   
1   242.0     42    -0.006613        0.018155       -0.014557        0.007888   
2   279.0     42    -0.010706        0.025654       -0.020854        0.011393   
3   354.0     42    -0.002826        0.019773       -0.014786        0.007011   
4   457.0     42    -0.009328        0.023650       -0.021518        0.011744   

   expense_beta_2        r2    f_stat  f_pvalue  
0        0.001754  0.094075  0.960559  0.440573  
1        0.004873  0.098202  1.007285  0.416168  
2        0.005486  0.109362  1.135811  0.354765  
3        0.009172  0.070100  0.697306  0.598692  
4        0.004549  0.107745  1.116989  0.363242  
Số bank estimate được: 5783


In [89]:
print(expense_beta_df_2["expense_beta_2"].describe().round(6))

count    5783.000000
mean        0.004032
std         0.012388
min        -0.151759
25%         0.002076
50%         0.004257
75%         0.006596
max         0.405310
Name: expense_beta_2, dtype: float64


# In Income beta

In [82]:
import pandas as pd
import numpy as np
final_df["totalinin"] = pd.to_numeric(final_df["RIAD4107"], errors="coerce")
final_df["avg_liab"] = pd.to_numeric(final_df["RCON2948"], errors="coerce")

final_df["avg_liab"] = final_df["avg_liab"].replace(0, np.nan)

# annualized interest expense rate
final_df["IntInRatebylia"] = 4 * (final_df["totalinexp"] / final_df["avg_liab"])

print(final_df[["RIAD4107", "avg_liab", "IntInRatebylia"]].head())

  RIAD4107  avg_liab  IntInRatebylia
0      830   67195.0        0.004465
1      425   38434.0        0.005828
2     1980  187680.0        0.009207
3       80    7281.0        0.001099
4      594   43452.0        0.005523


In [97]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# =========================
# 1. Chuẩn bị dữ liệu
# =========================
df_reg = final_df.copy()

# Chọn đúng biến phụ thuộc (ví dụ: sử dụng một biến khác thay vì 'IntInRatebylia')
y_col = "IntInRatebylia"  # Ví dụ: thay đổi thành biến bạn muốn sử dụng, ví dụ 'TOTAL_ASSETS'

# Ép kiểu
df_reg["IDRSSD"] = pd.to_numeric(df_reg["IDRSSD"], errors="coerce")
df_reg[y_col] = pd.to_numeric(df_reg[y_col], errors="coerce")
df_reg["EFFR"] = pd.to_numeric(df_reg["EFFR"], errors="coerce")

# Chuẩn hóa quarter
df_reg["quarter"] = df_reg["quarter"].astype(str).str.strip().str.upper()

# Bỏ dòng thiếu key
df_reg = df_reg.dropna(subset=["IDRSSD", "quarter", y_col, "EFFR"]).copy()

# Bỏ duplicate bank-quarter nếu có
df_reg = df_reg.drop_duplicates(subset=["IDRSSD", "quarter"])

# Tách year, quarter number để sort đúng
df_reg["year"] = df_reg["quarter"].str.extract(r"(\d{4})").astype(float)
df_reg["qnum"] = df_reg["quarter"].str.extract(r"Q([1-4])").astype(float)

df_reg = df_reg.dropna(subset=["year", "qnum"]).copy()
df_reg["year"] = df_reg["year"].astype(int)
df_reg["qnum"] = df_reg["qnum"].astype(int)

df_reg = df_reg.sort_values(["IDRSSD", "year", "qnum"]).reset_index(drop=True)

# =========================
# 2. Tạo sai phân biến phụ thuộc theo từng bank
# =========================
df_reg["d_" + y_col] = df_reg.groupby("IDRSSD")[y_col].diff()

# =========================
# 3. Tạo ΔEFFR theo quý + 3 lag
# =========================
effr_q = (
    df_reg[["quarter", "year", "qnum", "EFFR"]]
    .drop_duplicates(subset=["quarter"])
    .sort_values(["year", "qnum"])
    .reset_index(drop=True)
)

effr_q["d_EFFR"] = effr_q["EFFR"].diff()
effr_q["d_EFFR_l1"] = effr_q["d_EFFR"].shift(1)
effr_q["d_EFFR_l2"] = effr_q["d_EFFR"].shift(2)
effr_q["d_EFFR_l3"] = effr_q["d_EFFR"].shift(3)

df_reg = df_reg.drop(columns=["d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"], errors="ignore")

df_reg = df_reg.merge(
    effr_q[["quarter", "d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"]],
    on="quarter",
    how="left"
)

# =========================
# 4. Chạy OLS riêng cho từng bank
# =========================
results = []

for bank, g in df_reg.groupby("IDRSSD"):
    temp = g[[
        "d_" + y_col,
        "d_EFFR",
        "d_EFFR_l1",
        "d_EFFR_l2",
        "d_EFFR_l3"
    ]].dropna().copy()

    # Cần đủ quan sát
    if len(temp) < 8:
        continue

    # Y phải có biến thiên
    if temp["d_" + y_col].nunique() < 2:
        continue

    X = temp[["d_EFFR", "d_EFFR_l1", "d_EFFR_l2", "d_EFFR_l3"]]

    # X phải có biến thiên thực sự
    if (X.nunique() <= 1).any():
        continue

    y = temp["d_" + y_col]
    X = sm.add_constant(X)

    try:
        model = sm.OLS(y, X).fit()

        b0 = model.params.get("d_EFFR", np.nan)
        b1 = model.params.get("d_EFFR_l1", np.nan)
        b2 = model.params.get("d_EFFR_l2", np.nan)
        b3 = model.params.get("d_EFFR_l3", np.nan)

        income_beta = b0 + b1 + b2 + b3

        results.append({
            "IDRSSD": bank,
            "n_obs": len(temp),
            "coef_d_EFFR": b0,
            "coef_d_EFFR_l1": b1,
            "coef_d_EFFR_l2": b2,
            "coef_d_EFFR_l3": b3,
            "income_beta_2": income_beta,
            "r2": model.rsquared,
            "f_stat": model.fvalue,
            "f_pvalue": model.f_pvalue
        })

    except Exception as e:
        print(f"Skip bank {bank}: {e}")
        continue

# Đưa kết quả vào DataFrame
expense_income_df_2 = pd.DataFrame(results)

print(expense_income_df_2.head())
print("Số bank estimate được:", len(expense_income_df_2))

   IDRSSD  n_obs  coef_d_EFFR  coef_d_EFFR_l1  coef_d_EFFR_l2  coef_d_EFFR_l3  \
0    37.0     42    -0.004157        0.010268       -0.010827        0.006469   
1   242.0     42    -0.006613        0.018155       -0.014557        0.007888   
2   279.0     42    -0.010706        0.025654       -0.020854        0.011393   
3   354.0     42    -0.002826        0.019773       -0.014786        0.007011   
4   457.0     42    -0.009328        0.023650       -0.021518        0.011744   

   income_beta_2        r2    f_stat  f_pvalue  
0       0.001754  0.094075  0.960559  0.440573  
1       0.004873  0.098202  1.007285  0.416168  
2       0.005486  0.109362  1.135811  0.354765  
3       0.009172  0.070100  0.697306  0.598692  
4       0.004549  0.107745  1.116989  0.363242  
Số bank estimate được: 5783


In [98]:
print(expense_income_df_2.describe())


             IDRSSD        n_obs  coef_d_EFFR  coef_d_EFFR_l1  coef_d_EFFR_l2  \
count  5.783000e+03  5783.000000  5783.000000     5783.000000     5783.000000   
mean   1.028395e+06    36.464811    -0.013134        0.021983       -0.011212   
std    1.123425e+06    10.275861     0.026339        0.025233        0.017593   
min    3.700000e+01     8.000000    -1.327337       -0.232509       -0.471451   
25%    3.273700e+05    37.000000    -0.010257        0.012511       -0.017668   
50%    6.496320e+05    42.000000    -0.007574        0.018554       -0.012614   
75%    9.595365e+05    42.000000    -0.005322        0.025005       -0.006785   
max    5.860740e+06    42.000000     0.253287        1.238770        0.368427   

       coef_d_EFFR_l3  income_beta_2           r2       f_stat     f_pvalue  
count     5783.000000    5783.000000  5783.000000  5783.000000  5783.000000  
mean         0.006396       0.004032     0.168418     1.298072     0.367878  
std          0.013972       0.012388